# Static experiment: training, tuning and evaluation
Runs the same code as `python -m src.models.train --experiment static`.

In [ ]:
import sys, pathlib; sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
from src import config
import json
from src.preprocessing.loaders import load_static
from src.models.train import run_experiment

Training splits the data first, fits preprocessing on the training part only, tunes with cross-validation on the training part only, and touches the test set once.

In [ ]:
bundle = load_static(config.STATIC_CSV)
res = run_experiment('static', bundle)

In [ ]:
rows = []
for name, r in res['models'].items():
    for ver in ('baseline', 'tuned'):
        m = r[ver]; rows.append({'model': name, 'version': ver, **{k: round(m[k], 4) for k in ('accuracy','precision','recall','f1','false_positive_rate')}, 'roc_auc': m['roc_auc']})
pd.DataFrame(rows)

In [ ]:
print('Selected:', res['best_model'], '| rule:', res['selection_rule'])
for n, r in res['models'].items(): print(n, r['best_params'])

In [ ]:
import seaborn as sns, numpy as np
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, (n, r) in zip(axes, res['models'].items()):
    sns.heatmap(np.array(r['tuned']['confusion_matrix']), annot=True, fmt='d', cmap='Blues', ax=ax, xticklabels=['benign','malware'], yticklabels=['benign','malware'])
    ax.set(title=n, xlabel='Predicted', ylabel='True')
plt.tight_layout(); plt.show()

## Explainability

In [ ]:
pd.DataFrame(res['importance']['native']['items']).head(15), pd.DataFrame(res['importance']['permutation']['items']).head(15)

**Your discussion:** which model is best and why? Is recall or false-positive rate more important here? Which features matter and why does importance not mean causation? Any sign of overfitting (compare CV F1 with test F1)?